# S03 Exercises — Git, GitHub & Project Structure

**Python for Applied Physics** · Session 3 exercises  

These exercises are different from S01–S02: most of the work happens in the **terminal**, not in Python cells. Each exercise has a clear **goal**, a set of **terminal commands** to run, and a **verification step** (often a Python cell or a `git log` output) to confirm you got it right.

Difficulty: 🟢 accessible · 🟡 intermediate · 🔴 challenging

---

## Exercise 1 🟢 — Initialize your course repository

**Goal:** Create the `applied-physics-python` project repository, add all S01–S03 materials, and make your first commits.

**Steps:**

1. Create (or navigate to) your course project folder and initialize Git:
   ```bash
   mkdir applied-physics-python
   cd applied-physics-python
   git init
   ```

2. Run the `.gitignore` cell from the lecture notebook to create the ignore rules.

3. Stage and commit the `.gitignore` as the first commit:
   ```bash
   git add .gitignore
   git commit -m "Add .gitignore for scientific Python project"
   ```

4. Create the project structure folders and add the S01–S03 notebooks:
   ```bash
   mkdir -p S01_ecosystem/notebooks S02_control_flow/notebooks S03_git_github/notebooks shared
   # Copy your notebooks into the appropriate subfolders
   ```

5. Run the `shared/optics.py` and `requirements.txt` cells from the lecture to create those files.

6. Stage everything and commit:
   ```bash
   git add .
   git commit -m "Add S01-S03 notebooks, shared optics module, requirements"
   ```

**Verify:** Run the cell below — it should show your two commits.

In [2]:
import subprocess
result = subprocess.run(["git", "log", "--oneline"], capture_output=True, text=True)
print(result.stdout if result.stdout else result.stderr)

19f2116 Add S01-S03 notebooks, shared optics module, requirements
5161261 Add .gitignore for scientific Python project



---
## Exercise 2 🟢 — Commit message practice

**Goal:** Understand what makes a good commit message by examining a real project's history.

**Steps:**

1. Clone the NumPy repository (a real large scientific Python project) into a temporary location:
   ```bash
   cd /tmp
   git clone --depth 50 https://github.com/numpy/numpy.git numpy-temp
   cd numpy-temp
   git log --oneline -20
   ```
   The `--depth 50` flag clones only the last 50 commits (faster).

2. Run the cell below to fetch the last 20 NumPy commit messages programmatically and classify them.

3. Answer in the Markdown cell at the bottom: what patterns do you observe in good scientific open-source commit messages?

In [ ]:
import subprocess
import os

numpy_path = "/tmp/numpy-temp"

if os.path.exists(numpy_path):
    result = subprocess.run(
        ["git", "log", "--oneline", "-20"],
        capture_output=True, text=True, cwd=numpy_path
    )
    lines = result.stdout.strip().split("\n")
    print(f"Last {len(lines)} NumPy commits:\n")
    for i, line in enumerate(lines, 1):
        print(f"  {i:2d}. {line}")
else:
    print("numpy-temp not found. Run the clone command in the terminal first.")

**Your observations about good commit messages:**

*(Write 3–5 observations here)*

---
## Exercise 3 🟢 — Push to GitHub

**Goal:** Put your `applied-physics-python` repository on GitHub.

**Steps:**

1. If you haven't already, set up SSH authentication (see lecture Section 4.1).

2. On GitHub, create a new **empty** repository named `applied-physics-python` (do NOT initialize with a README or `.gitignore`).

3. Add the remote and push:
   ```bash
   git remote add origin git@github.com:YOUR_USERNAME/applied-physics-python.git
   git push -u origin main
   ```

4. Open your repository URL in the browser and confirm the files are there.

**Verify:** Run the cell below.

In [1]:
import subprocess
result = subprocess.run(["git", "remote", "-v"], capture_output=True, text=True)
print(result.stdout if result.stdout else "No remotes configured yet.")

No remotes configured yet.


---
## Exercise 4 🟡 — Feature branch workflow

**Goal:** Use a branch to add a new function to `shared/optics.py` without touching `main` until it's ready.

**The function to add:** `peak_irradiance(E_J, tau_s, w0_m)` — the corrected version from S02 Exercise 8.

```
I_peak = 2 * E / (π * w0² * τ)    [W/m²]
```

**Steps:**

1. Create and switch to a feature branch:
   ```bash
   git switch -c feature/peak-irradiance
   ```

2. Add the `peak_irradiance` function to `shared/optics.py`. Include a proper NumPy-style docstring.

3. Commit on the feature branch:
   ```bash
   git add shared/optics.py
   git commit -m "Add peak_irradiance function to optics module"
   ```

4. Switch back to `main` and confirm `peak_irradiance` is NOT there yet:
   ```bash
   git switch main
   ```

5. Merge and delete the branch:
   ```bash
   git merge feature/peak-irradiance
   git branch -d feature/peak-irradiance
   ```

6. Push the updated `main` to GitHub:
   ```bash
   git push
   ```

**Verify:** Run the cell below.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "..", "shared"))

# Force reload in case the module was already imported
import importlib
import optics
importlib.reload(optics)

I = optics.peak_irradiance(1e-6, 300e-15, 50e-6)
print(f"I_peak = {I:.3e} W/m²  (expected ~8.49e11 W/m²)")

---
## Exercise 5 🟡 — Simulating and resolving a merge conflict

**Goal:** Deliberately create a merge conflict and resolve it — in a safe sandbox.

**Steps:**

1. Create a new branch:
   ```bash
   git switch -c experiment/docstring-style
   ```

2. In `shared/optics.py`, change the docstring of `rayleigh_range` to use Google style instead of NumPy style. Commit:
   ```bash
   git commit -am "Try Google-style docstrings in rayleigh_range"
   ```

3. Switch back to `main` and make a *different* change to the same docstring (e.g., add a `Notes` section). Commit:
   ```bash
   git switch main
   # Edit shared/optics.py — change the rayleigh_range docstring differently
   git commit -am "Add Notes section to rayleigh_range docstring"
   ```

4. Now try to merge:
   ```bash
   git merge experiment/docstring-style
   ```
   Git will report a conflict.

5. Open `shared/optics.py` in VS Code. Use the merge editor to resolve the conflict — keep the NumPy style with the Notes section.

6. Stage and complete the merge:
   ```bash
   git add shared/optics.py
   git commit -m "Resolve merge conflict: keep NumPy docstring style with Notes"
   git branch -d experiment/docstring-style
   ```

**Verify:** Run the cell below.

In [ ]:
import subprocess
# Check there are no conflict markers left in the file
with open(os.path.join("..", "..", "shared", "optics.py")) as f:
    content = f.read()

if "<<<<<<<" in content or "=======" in content or ">>>>>>>" in content:
    print("⚠️  Conflict markers still present! Resolve and recommit.")
else:
    print("✅  No conflict markers found. Merge resolved cleanly.")
    result = subprocess.run(["git", "log", "--oneline", "-5"], capture_output=True, text=True)
    print("\nLast 5 commits:")
    print(result.stdout)

---
## Exercise 6 🟡 — Add a `shared/pulses.py` module

**Goal:** Migrate the `pulse_parameters` function from S02 into a new `shared/pulses.py` module using the feature branch workflow.

**Requirements for `pulses.py`:**
1. `pulse_parameters(E_J, tau_s, rep_rate_Hz)` — as written in S02 (peak power, avg power, duty cycle)
2. `transform_limit(delta_lambda_nm, center_lambda_nm)` — the corrected version from S02 Exercise 8
3. A module-level docstring explaining the module's purpose
4. All functions with NumPy-style docstrings

**Workflow requirements:**
- Use a feature branch named `feature/pulses-module`
- Make at least two commits on the branch (e.g., one per function)
- Merge back to `main` and push

**Verify:**

In [ ]:
import importlib, sys, os
sys.path.insert(0, os.path.join("..", "..", "shared"))

import pulses
importlib.reload(pulses)

# Test pulse_parameters
P_peak, P_avg, dc = pulses.pulse_parameters(50e-9, 300e-15, 76e6)
print(f"Peak power    : {P_peak/1e3:.2f} kW   (expected ~157 kW)")
print(f"Average power : {P_avg:.3f} W    (expected 3.800 W)")

# Test transform_limit
dt = pulses.transform_limit(10, 800)
print(f"Transform limit: {dt:.1f} fs   (expected ~94 fs)")

---
## Exercise 7 🔴 — Repo archaeology: exploring git history programmatically

**Goal:** Use Python's `subprocess` module to extract information from a Git repository's history — a useful skill for automated reporting and reproducibility checks.

Write a function `repo_report(repo_path)` that:
1. Finds the total number of commits
2. Finds the date of the first and last commit
3. Finds the author with the most commits
4. Lists all files currently tracked by Git (not ignored)
5. Reports the number of Python files tracked
6. Returns all results as a dictionary

Use only `subprocess.run` with `git` commands. Useful commands:
- `git rev-list --count HEAD` — total commits
- `git log --format="%ai" --reverse` — all commit dates (oldest first)
- `git log --format="%an"` — author name per commit
- `git ls-files` — all tracked files

Test it on your `applied-physics-python` repo and on the NumPy clone in `/tmp/numpy-temp` (if available).

In [ ]:
import subprocess
from collections import Counter

def repo_report(repo_path):
    """
    Generate a summary report of a Git repository.

    Parameters
    ----------
    repo_path : str
        Absolute or relative path to the repository root.

    Returns
    -------
    dict
        Repository statistics.
    """
    # Your code here
    pass


# Test on your repo
report = repo_report(os.path.join("..", ".."))
if report:
    for key, val in report.items():
        print(f"  {key}: {val}")

---
## Exercise 8 🔴 — Write a `README.md` for your repository

**Goal:** Write a proper README for your `applied-physics-python` repository. A README is often the first thing a collaborator (or a future employer) reads.

**Requirements:**
Your README must include the following sections, written in Markdown:
1. **Title and one-sentence description**
2. **Course context** — which course, for whom, what it covers
3. **Setup** — how to clone, create a virtual environment, and install dependencies
4. **Repository structure** — a tree showing the folder layout with a one-line description of each folder
5. **Session overview** — a table listing S01–S03 with their topics
6. **How to run the notebooks** — one-line instructions

Write the README in the cell below (it generates the file), then commit and push it.

> **💡 Pro Tip:** GitHub renders Markdown automatically. After pushing, visit your repo page — the README appears as the landing page.

In [ ]:
readme_content = """
# Applied Physics Python Course

<!-- Your README here — replace this placeholder with the real content -->

"""

readme_path = os.path.join("..", "..", "README.md")
with open(readme_path, "w") as f:
    f.write(readme_content.strip())

print(f"README.md written to {os.path.abspath(readme_path)}")
print("Now commit and push:")
print("  git add README.md")
print('  git commit -m "Add project README with setup and structure guide"')
print("  git push")